In [2]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [3]:
# features = joblib.load('../../predictors/hist/feats_gk')

features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "percentage_net_transfers",
    "elo_diff",
    "points_5_ewm",
    "xP_5_ewm",
    "minutes_rolling_3",
    "clean_sheets_3_ewm",
    "goals_conceded_3_ewm",
    "yellow_cards_rolling_5",
    "saves_rolling_5",
    "ict_index_rolling_3",
    "starts_rolling_3",
    "shots_faced_3_ewm",
    "points_3_ewm",
    "saves_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_conceded_3_ewm",
    "goals_prevented_3_ewm",
    "opponent_xG_3_ewm",
    "selected_by_percent",
    "transfers_in",
]


player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Goalkeeper"]

league_models = {}

for gw in range(21, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]

    # feats.remove('xP')
    # print('--------', feats)
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.7,
        min_corr=0.1,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="GK", random_seed=42
    )

    print(f"Features for GK kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model


Initializing NUTS using jitter+adapt_diag...



 Model for round 21 
  dropping 'points_3_ewm' (|corr| with xP=0.875) -- |corr|=0.981 with already-selected 'points_5_ewm'
  dropping 'xP_5_ewm' (|corr| with xP=0.829) -- |corr|=0.908 with already-selected 'points_5_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.811) -- |corr|=0.905 with already-selected 'points_5_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.809) -- |corr|=0.904 with already-selected 'points_5_ewm'
  dropping 'ict_index_rolling_3' (|corr| with xP=0.767) -- |corr|=0.867 with already-selected 'points_5_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.766) -- |corr|=0.869 with already-selected 'points_5_ewm'
  dropping 'saves_3_ewm' (|corr| with xP=0.732) -- |corr|=0.834 with already-selected 'points_5_ewm'
  dropping 'clean_sheets_3_ewm' (|corr| with xP=0.706) -- |corr|=0.798 with already-selected 'points_5_ewm'
  dropping 'expected_goals_conceded_3_ewm' (|corr| with xP=0.685) -- |corr|=0.764 with already-selected 'points_5_ewm'
  dropping 'saves_rolli

Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 28 seconds.
Initializing NUTS using jitter+adapt_diag...


  divergences: 0
  dropping 'goals_prevented_3_ewm' (label beta[7]) -- HDI near-duplicate of 'percentage_net_transfers' (label beta[2])
Kept 7 / 10 features: ['difficulty', 'percentage_net_transfers', 'elo_diff', 'points_5_ewm', 'goals_conceded_3_ewm', 'yellow_cards_rolling_5', 'selected_by_percent']
Refitting on pruned feature set...


... (138 outputs left out)

Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 49 seconds.


  divergences: 0
Features for GK kept in gw 38 -> ['percentage_net_transfers', 'points_5_ewm', 'yellow_cards_rolling_5', 'selected_by_percent', 'transfers_in']


In [4]:
joblib.dump(league_models, "./league_models_gk")

['./league_models_gk']